<a href="https://colab.research.google.com/github/SHRESHTH121/Demo/blob/main/Aadhaar_Validator_Robust.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install easyocr pyzbar opencv-python-headless --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.6/299.6 kB 12.8 MB/s eta 0:00:00


In [2]:
import cv2
import easyocr
import re
import numpy as np
import os
import json
from google.colab import files

In [3]:
print("Upload Front Aadhaar Image")
front_upload = files.upload()

print("Upload Back Aadhaar Image")
back_upload = files.upload()

front_image_path = next(iter(front_upload))
back_image_path  = next(iter(back_upload))

print("Front:", front_image_path)
print("Back :", back_image_path)

Upload Front Aadhaar Image


Saving adhaar front side.jpeg to adhaar front side.jpeg
Upload Back Aadhaar Image


Saving adhaar backside.jpeg to adhaar backside.jpeg
Front: adhaar front side.jpeg
Back : adhaar backside.jpeg


In [4]:
# Load EasyOCR with English (+ Hindi for transliterated names / addresses)
reader = easyocr.Reader(['en', 'hi'], gpu=False)

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

# **Document Validation**

In [6]:
def preprocess_for_detection(image_path):
    """
    Read image and apply CLAHE + adaptive thresholding so
    detection works on dark / washed-out / low-contrast scans.
    Returns (img_bgr, img_gray, img_enhanced_gray) or raises.
    """
    img = cv2.imread(image_path)
    if img is None:
        raise IOError(f"Cannot read image: {image_path}")

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # CLAHE equalises local contrast – helps on phone-camera shots
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)

    return img, gray, enhanced


def card_fits_frame(
    image_path: str,
    corner_pct: float = 0.08,       # slightly smaller ROI → less harsh
    min_corner_std: float = 6.0,    # lowered; dark backgrounds can have std ~7
    max_tilt_deg: float = 20.0,     # real-world phone shots tilt up to ~18°
):
    """
    Check that the card fills the frame and is roughly level.

    Improvements over original:
    • CLAHE preprocessing before edge / tilt analysis
    • Checks ALL four corners (not just weakest) with a vote
    • Tilt uses Otsu-thresholded edges + HoughLinesP for cleaner lines
    • Falls back gracefully when Hough finds no lines
    """
    try:
        img, gray, enhanced = preprocess_for_detection(image_path)
    except IOError as e:
        return False, str(e)

    H, W = img.shape[:2]
    ch = max(1, int(H * corner_pct))
    cw = max(1, int(W * corner_pct))

    corner_patches = {
        "top-left":     enhanced[:ch, :cw],
        "top-right":    enhanced[:ch, -cw:],
        "bottom-left":  enhanced[-ch:, :cw],
        "bottom-right": enhanced[-ch:, -cw:],
    }

    corner_stds = {k: float(v.std()) for k, v in corner_patches.items()}

    # Use majority-vote: fail only if 3+ corners look like bare background
    low_contrast_corners = [k for k, s in corner_stds.items() if s < min_corner_std]
    if len(low_contrast_corners) >= 3:
        return False, (
            f"Card does not fill frame – background visible in: "
            f"{', '.join(low_contrast_corners)} "
            f"(stds={[round(corner_stds[k],1) for k in low_contrast_corners]})"
        )

    # --- Tilt detection ---
    scale = min(1.0, 600.0 / max(H, W))
    small_gray = cv2.resize(enhanced, (int(W * scale), int(H * scale)))

    # Otsu threshold → cleaner edges than fixed Canny thresholds
    _, otsu = cv2.threshold(small_gray, 0, 255,
                            cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    edges = cv2.Canny(otsu, 50, 150)

    lines = cv2.HoughLinesP(
        edges, 1, np.pi / 180,
        threshold=40,
        minLineLength=int(min(H, W) * scale * 0.10),  # 10% of shorter side
        maxLineGap=15,
    )

    tilt = 0.0
    if lines is not None and len(lines) > 0:
        angles = []
        for line in lines:
            x1, y1, x2, y2 = line[0]
            if x2 == x1:
                continue
            angle = np.degrees(np.arctan2(y2 - y1, x2 - x1)) % 180
            if angle > 90:
                angle -= 180
            if angle > 45:
                angle = 90 - angle
            angles.append(abs(angle))
        if angles:
            tilt = float(np.percentile(angles, 25))  # 25th-pct is more robust than median

    if tilt > max_tilt_deg:
        return False, f"Card tilted by {tilt:.1f}° (limit {max_tilt_deg}°)"

    weakest_std = min(corner_stds.values())
    return True, (
        f"Card fits frame (weakest_corner_std={weakest_std:.1f}, tilt={tilt:.1f}°)"
    )


In [7]:
for label, path in [("Front", front_image_path), ("Back", back_image_path)]:
    ok, msg = card_fits_frame(path)
    print(f"{label}: {'PASS ✓' if ok else 'FAIL ✗'}  —  {msg}")

Front: PASS ✓  —  Card fits frame (weakest_corner_std=46.0, tilt=0.0°)
Back: PASS ✓  —  Card fits frame (weakest_corner_std=51.1, tilt=1.2°)


# **Blur Detection**

In [8]:
def calculate_blur_score(image_path):
    """
    Laplacian variance with tiling.
    For flat-design digital Aadhaar cards (new PVC format), the card face has
    very low texture by design — the text and image areas must be scored
    separately, otherwise the large plain background drags the score down.
    """
    img, gray, enhanced = preprocess_for_detection(image_path)
    H, W = gray.shape

    global_score = float(cv2.Laplacian(gray, cv2.CV_64F).var())

    # Tiled scoring: 3x3 grid
    tile_scores = []
    for ri in range(3):
        for ci in range(3):
            tile = gray[ri*H//3:(ri+1)*H//3, ci*W//3:(ci+1)*W//3]
            tile_scores.append(float(cv2.Laplacian(tile, cv2.CV_64F).var()))

    # Mean of top-3 tiles (not min) — avoids penalising plain-background tiles
    top3_mean = float(np.mean(sorted(tile_scores, reverse=True)[:3]))

    # Composite: weight global 50%, top-tile mean 50%
    composite = 0.5 * global_score + 0.5 * top3_mean
    return {
        "global":    round(global_score, 2),
        "top3_mean": round(top3_mean, 2),
        "composite": round(composite, 2),
    }


def analyze_aadhaar_blur(
    front_image_path,
    back_image_path,
    front_threshold=35,   # NEW: lower for flat-design PVC/digital cards
    back_threshold=55,    # NEW: back has QR which inflates score anyway
):
    """
    Separate thresholds for front and back:
      - Front: low natural texture (plain background, clean print) → lower bar
      - Back:  QR code adds a lot of edge energy → higher bar is fine
    Also flags when score is implausibly high (likely a photo of a screen/photocopy).
    """
    results = {}
    for label, path, threshold in [
        ("front", front_image_path, front_threshold),
        ("back",  back_image_path,  back_threshold),
    ]:
        scores = calculate_blur_score(path)
        is_blurry = scores["composite"] < threshold
        # Sanity-check: very high score (>2000) on a real photo usually means
        # the image is a scan/photocopy with moire patterns — warn but don't fail
        suspicious_sharp = scores["global"] > 2000
        results[label] = {
            "image_path": path,
            **scores,
            "threshold":  threshold,
            "status":     "Blurry" if is_blurry else "Clear",
            "is_blurry":  is_blurry,
            "suspicious_oversharp": suspicious_sharp,
        }
    return results


In [9]:
blur_report = analyze_aadhaar_blur(
    front_image_path, back_image_path,
    front_threshold=35,   # flat-design PVC cards have low texture by nature
    back_threshold=55,
)
print("\n========== BLUR REPORT ==========\n")
for side, info in blur_report.items():
    status = info['status']
    print(f"{side.upper()}: {status}  "
          f"(composite={info['composite']}, global={info['global']}, "
          f"top3_mean={info['top3_mean']})")
    if info.get('suspicious_oversharp'):
        print(f"  ⚠ Suspiciously high sharpness — may be a scan or photocopy")


========== BLUR REPORT ==========

FRONT: Clear  (composite=111.91, global=73.18, top3_mean=150.63)
BACK: Clear  (composite=165.89, global=123.43, top3_mean=208.36)


# **OCR Extraction**

In [10]:
def extract_text(image_path, min_conf=0.25):
    """
    Run EasyOCR and return a plain-text string.

    Improvements:
    • confidence filter  – drops garbage low-confidence tokens
    • deduplication      – consecutive identical lines removed
    • strips stray punctuation artifacts OCR often introduces
    """
    results = reader.readtext(image_path, detail=1, paragraph=False)

    lines = []
    for (_, text, conf) in results:
        if conf < min_conf:
            continue
        text = text.strip()
        # Remove lines that are purely punctuation / noise
        if re.fullmatch(r'[^A-Za-z0-9\u0900-\u097F]+', text):
            continue
        if not lines or text.lower() != lines[-1].lower():  # dedup
            lines.append(text)

    return "\n".join(lines)


In [11]:
def extract_aadhaar_number(text):
    """
    Matches all common OCR representations:
      • 'XXXX XXXX XXXX'  (spaces)
      • 'XXXX-XXXX-XXXX'  (hyphens)
      • 'XXXXXXXXXXXXXXXX'  (no separator)
    Also auto-corrects common OCR mis-reads: O→0, I/l→1, S→5, B→8
    """
    # Normalise common OCR character confusions before matching
    ocr_fix = str.maketrans("OIlSB", "01158")
    text_fixed = text.translate(ocr_fix)

    # Try structured formats first (more reliable)
    patterns = [
        r'\b(\d{4}[\s\-]\d{4}[\s\-]\d{4})\b',   # with separator
        r'\b(\d{12})\b',                            # no separator
    ]
    for pat in patterns:
        for match in re.finditer(pat, text_fixed):
            candidate = re.sub(r'[\s\-]', '', match.group(1))
            # Aadhaar cannot start with 0 or 1
            if len(candidate) == 12 and candidate[0] not in ('0', '1'):
                # Return original spacing style from the fixed text
                return match.group(1)
    return None


In [12]:
def extract_dob(text):
    """
    Handles multiple date formats and common OCR errors on date separators.
    Formats: DD/MM/YYYY, DD-MM-YYYY, DD.MM.YYYY
    Also matches 'DOB:' / 'Year of Birth:' prefixes.
    """
    # Normalise OCR artefacts around separators (e.g. 'O' instead of '0')
    text_fixed = re.sub(r'(?<=[0-9])[Oo](?=[0-9])', '0', text)  # O between digits → 0

    patterns = [
        r'(?:DOB|Date of Birth|D\.O\.B\.?)\s*[:\-]?\s*(\d{2}[/.\-]\d{2}[/.\-]\d{4})',
        r'\b(\d{2}[/.\-]\d{2}[/.\-]\d{4})\b',
        r'(?:Year of Birth|YOB)\s*[:\-]?\s*(\d{4})',  # fallback: year only
    ]
    for pat in patterns:
        m = re.search(pat, text_fixed, re.IGNORECASE)
        if m:
            return m.group(1)
    return None


In [13]:
def extract_gender(text):
    """
    Case-insensitive substring search.
    Handles OCR variants like 'Mal e', 'FEMAL E', 'mahila' (Hindi).
    """
    text_lower = text.lower().replace(' ', '')  # collapse spaces (OCR splits words)

    # Map keywords → canonical gender
    gender_map = [
        (r'female|femal|महिला', 'Female'),
        (r'male|purus|पुरुष',   'Male'),
        (r'transgender|तृतीय', 'Transgender'),
    ]
    for pattern, label in gender_map:
        if re.search(pattern, text_lower):
            return label
    return None


In [14]:
def extract_name(text):
    """
    Finds the cardholder's name on the front face.

    Strategy:
    1. Look for a line immediately after 'Government of India' header
       (Aadhaar cards print name 1-2 lines after this header).
    2. Fall back to the first clean alphabetic multi-word line that
       doesn't match known noise tokens.
    """
    lines = [l.strip() for l in text.split('\n') if l.strip()]

    NOISE = {
        'government', 'india', 'dob', 'male', 'female', 'transgender',
        'aadhaar', 'address', 'uid', 'uidai', 'enrolment', 'vid',
        'unique', 'identification', 'authority', 'help', 'www', 'of',
        'download', 'digitally', 'signed', 'verified',
    }

    def is_valid_name_line(line):
        if len(line.split()) < 2:
            return False
        if re.search(r'\d', line):
            return False
        if any(w in line.lower() for w in NOISE):
            return False
        # Must be mostly Latin letters (Aadhaar prints name in English on front)
        alpha_ratio = sum(c.isalpha() for c in line) / max(len(line), 1)
        if alpha_ratio < 0.70:
            return False
        return True

    # Strategy 1: look for line after 'Government of India'
    for i, line in enumerate(lines):
        if re.search(r'government\s+of\s+india', line, re.IGNORECASE):
            for candidate in lines[i+1 : i+4]:
                if is_valid_name_line(candidate):
                    return candidate
            break

    # Strategy 2: first clean multi-word alphabetic line
    for line in lines:
        if is_valid_name_line(line):
            return line

    return None


In [15]:
def extract_address(text):
    """
    Handles two layouts that trip up simple line-by-line parsing:

    Layout A (old cards): 'Address:' on its own line, then address on next lines
    Layout B (new PVC):   'Address: S/O: Name, house, street …' all on one line
                          OR OCR merges 'Address:' with the first address line

    Also handles the vertical 'Details as on:' watermark that EasyOCR picks up
    on the left edge of new-format backs — this would otherwise be the first
    'line' and confuse the start-trigger.
    """
    # ── Pre-clean: remove the rotated watermark OCR artifact ──────────────────
    text = re.sub(
        r'Details\s+as\s+on\s*[:\-]?\s*[\d/]+',
        '',
        text,
        flags=re.IGNORECASE,
    )

    # ── Split into tokens on both newlines AND certain punctuation separators ──
    # EasyOCR sometimes returns the entire address as one long string.
    # Break it at 'S/O', 'D/O', 'W/O', 'C/O' in addition to newlines so the
    # start-trigger can fire even when the label and content are merged.
    text = re.sub(r'\b([SDWC]/O\s*:)', r'\n\1', text, flags=re.IGNORECASE)
    text = re.sub(r'\bAddress\s*:', r'\nAddress:\n', text, flags=re.IGNORECASE)

    lines = [l.strip() for l in text.split('\n') if l.strip()]

    START_PATTERNS = [
        r'\baddr(ess)?\b',
        r'\bc/o\b', r'\bs/o\b', r'\bd/o\b', r'\bw/o\b',
        r'\bhouse\b', r'\bflat\b', r'\bplot\b', r'\bvill(age)?\b',
        r'#\s*\d',        # "#535/2" style house numbers
        r'\bward\b', r'\bnear\b', r'\bsector\b',
    ]
    STOP_PATTERNS = [
        r'\bvid\b', r'uidai', r'www\.', r'\b1947\b',
        r'help\s*centre', r'toll[\s\-]?free', r'download',
        r'unique\s+id', r'help@',
    ]

    collecting = False
    address_lines = []

    for line in lines:
        line_l = line.lower()

        if collecting and any(re.search(p, line_l) for p in STOP_PATTERNS):
            break
        if collecting and len(address_lines) >= 10:
            break

        if not collecting and any(re.search(p, line_l) for p in START_PATTERNS):
            collecting = True
            # If this line is ONLY the label 'Address:', skip it
            if re.fullmatch(r'addr(ess)?\s*:?', line_l):
                continue

        if collecting:
            # Skip lines that are just the Aadhaar / VID number
            if re.search(r'\b\d{4}\s?\d{4}\s?\d{4}\b', line):
                continue
            address_lines.append(line)

    # ── Last resort: if nothing found, look for pincode and grab surrounding text ──
    if not address_lines:
        full = ' '.join(lines)
        m = re.search(r'(.{0,200})\b(\d{6})\b', full)
        if m:
            return m.group(0).strip()

    return '\n'.join(address_lines).strip()


In [16]:
def detect_photo(image_path):
    """
    Multi-scale face detection with fallback to profile-face cascade.
    Also tries on the enhanced (CLAHE) image when the raw image fails.
    """
    img, gray, enhanced = preprocess_for_detection(image_path)

    cascade_paths = [
        cv2.data.haarcascades + 'haarcascade_frontalface_default.xml',
        cv2.data.haarcascades + 'haarcascade_frontalface_alt2.xml',
        cv2.data.haarcascades + 'haarcascade_profileface.xml',
    ]

    for cp in cascade_paths:
        det = cv2.CascadeClassifier(cp)
        if det.empty():
            continue
        for image_to_try in [gray, enhanced]:
            faces = det.detectMultiScale(
                image_to_try,
                scaleFactor=1.05,    # finer scale steps → fewer misses
                minNeighbors=3,      # slightly more permissive
                minSize=(30, 30),
                flags=cv2.CASCADE_SCALE_IMAGE,
            )
            if len(faces) > 0:
                return True

    return False


In [17]:
def _sharpen_for_qr(gray):
    """
    Unsharp-mask sharpening tuned for JPEG-compressed QR modules.
    Makes module boundaries crisper so decoder can lock on finder patterns.
    """
    blur = cv2.GaussianBlur(gray, (0, 0), 2.0)
    sharp = cv2.addWeighted(gray, 1.8, blur, -0.8, 0)
    return sharp


def _try_decode_variants(detector, image):
    """Try raw, CLAHE-enhanced, and sharpened variants on one detector call."""
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(image)
    sharpened = _sharpen_for_qr(image)
    binarized = cv2.adaptiveThreshold(
        image, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 11, 2,
    )
    for variant_name, variant in [
        ("raw", image),
        ("enhanced", enhanced),
        ("sharpened", sharpened),
        ("binarized", binarized),
    ]:
        data, bbox, _ = detector.detectAndDecode(variant)
        if bbox is not None and data:
            return data, variant_name
    return None, None


def detect_qr_presence(image_path):
    """
    Robust four-tier QR detection for Aadhaar cards.

    IMPORTANT: Aadhaar back QR codes are UIDAI Secure QR codes — they contain
    XML-encoded, digitally-signed personal data compressed and encoded as a QR.
    Standard decoders (pyzbar, cv2) fail on them when the image has JPEG
    compression artifacts at module boundaries. The fix is aggressive sharpening
    + binarisation before decode, and trying multiple image regions.

    Tier 1: cv2.QRCodeDetector  (full image + cropped quadrants + sharpened)
    Tier 2: cv2.wechat_qrcode   (CNN-based, far better on low-quality QRs)
    Tier 3: pyzbar              (good on clean images)
    Tier 4: Morphological heuristic (presence-only, no decode)
    """
    try:
        img, gray, enhanced = preprocess_for_detection(image_path)
    except IOError as e:
        return {"qr_present": False, "reason": str(e)}

    H, W = img.shape[:2]

    # ── Tier 1: cv2.QRCodeDetector on full image + quadrants ──────────────────
    qr_det = cv2.QRCodeDetector()
    data, variant = _try_decode_variants(qr_det, gray)
    if data:
        return {"qr_present": True, "method": f"cv2.QRCodeDetector({variant})",
                "data_length": len(data)}

    # Try on each image quadrant (QR might be in a corner)
    quadrants = {
        "TL": gray[:H//2, :W//2], "TR": gray[:H//2, W//2:],
        "BL": gray[H//2:, :W//2], "BR": gray[H//2:, W//2:],
    }
    for qname, quad in quadrants.items():
        # Upscale small quadrant so QR modules are larger
        quad_up = cv2.resize(quad, None, fx=1.5, fy=1.5, interpolation=cv2.INTER_CUBIC)
        data, variant = _try_decode_variants(qr_det, quad_up)
        if data:
            return {"qr_present": True,
                    "method": f"cv2.QRCodeDetector({qname},{variant})",
                    "data_length": len(data)}

    # ── Tier 2: WeChatQR (much better CNN-based finder) ───────────────────────
    try:
        wechat_det = cv2.wechat_qrcode_WeChatQRCode()
        for image_to_try in [gray, enhanced, _sharpen_for_qr(gray)]:
            texts, points = wechat_det.detectAndDecode(image_to_try)
            if texts:
                return {"qr_present": True, "method": "WeChatQR",
                        "data_length": len(texts[0])}
    except (cv2.error, AttributeError):
        pass  # WeChatQR not available in all builds

    # ── Tier 3: pyzbar ────────────────────────────────────────────────────────
    try:
        from pyzbar.pyzbar import decode as pyz_decode
        for image_to_try in [img, cv2.cvtColor(_sharpen_for_qr(gray), cv2.COLOR_GRAY2BGR)]:
            decoded = pyz_decode(image_to_try)
            if decoded:
                sym = decoded[0]
                return {"qr_present": True, "method": f"pyzbar({sym.type})",
                        "data_length": len(sym.data)}
    except ImportError:
        pass

    # ── Tier 4: Morphological heuristic (PRESENCE only) ───────────────────────
    # Aadhaar secure QR is always in the top-right of back or bottom-right of front.
    # Use Otsu + morphological closing to locate a dense square region.
    blurred = cv2.GaussianBlur(enhanced, (5, 5), 0)
    _, bw = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    closed = cv2.morphologyEx(bw, cv2.MORPH_CLOSE, kernel, iterations=2)

    contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    image_area = H * W

    # Also explicitly check the four corners where QR typically lives
    corner_regions = [
        ("TR", gray[: H//3, W//2 :]),       # back-card QR
        ("BR", gray[H//2 :, W//2 :]),        # front new-format QR
        ("BL", gray[H//2 :, : W//2]),
    ]
    for cname, region in corner_regions:
        if region.size == 0:
            continue
        rH, rW = region.shape
        sharp_r = _sharpen_for_qr(region)
        _, rbw = cv2.threshold(sharp_r, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        density = float(rbw.mean()) / 255.0
        # A QR corner region has 30-65% black pixels and >10 Laplacian var per unit area
        lap = float(cv2.Laplacian(region, cv2.CV_64F).var())
        if 0.25 <= density <= 0.68 and lap > 50:
            return {
                "qr_present": True,
                "method":     f"morphological_corner({cname})",
                "density":    round(density, 3),
                "lap_var":    round(lap, 1),
            }

    # Global contour scan
    for cnt in sorted(contours, key=cv2.contourArea, reverse=True)[:25]:
        area = cv2.contourArea(cnt)
        if area < 0.008 * image_area or area > 0.40 * image_area:
            continue
        x, y, w, h = cv2.boundingRect(cnt)
        aspect = w / float(h)
        if not (0.72 <= aspect <= 1.38):
            continue
        roi = bw[y:y+h, x:x+w]
        density = roi.mean() / 255.0
        if 0.22 <= density <= 0.72:
            return {
                "qr_present": True,
                "method":     "morphological_global",
                "bbox":       (x, y, w, h),
                "density":    round(density, 3),
            }

    return {"qr_present": False, "reason": "No QR code detected by any method"}


In [18]:
def extract_aadhaar_information(front_image, back_image):
    front_text = extract_text(front_image)
    back_text  = extract_text(back_image)

    aadhaar_data = {
        "front_text":     front_text,
        "back_text":      back_text,
        "name":           extract_name(front_text),
        "dob":            extract_dob(front_text),
        "gender":         extract_gender(front_text),
        "aadhaar_number": extract_aadhaar_number(front_text),
        "address":        extract_address(back_text),
        "photo_present":  detect_photo(front_image),
        "qr_present":     detect_qr_presence(back_image),
    }
    return aadhaar_data


In [19]:
def validate_completeness(aadhaar_data):
    mandatory = ["name", "dob", "gender", "aadhaar_number", "address"]
    missing = []
    for field in mandatory:
        val = aadhaar_data.get(field)
        # Treat empty strings and whitespace-only as missing too
        if not val or (isinstance(val, str) and not val.strip()):
            missing.append(field)
    return missing


In [20]:
aadhaar_data = extract_aadhaar_information(front_image_path, back_image_path)
missing_fields = validate_completeness(aadhaar_data)

print("\n========== EXTRACTED DATA ==========\n")
for k, v in aadhaar_data.items():
    if k not in ('front_text', 'back_text'):
        print(f"  {k}: {v}")

print("\nMissing fields:", missing_fields if missing_fields else "None ✓")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



========== EXTRACTED DATA ==========

  name: Shreshth Garg
  dob: 13/07/2005
  gender: Male
  aadhaar_number: 7445 3447 0565
  address: Acaress: SO:Padcep Kumar = ५३५/२. randhawa voad nard no  ७ Kharar PO: Kharar DIST: SAS Nagar Mohali). Puniab 140301
  photo_present: True
  qr_present: {'qr_present': True, 'method': 'morphological_corner(TR)', 'density': 0.352, 'lap_var': 111.3}

Missing fields: None ✓


# **Aadhaar Number Validation (Verhoeff)**

In [22]:
# Verhoeff algorithm tables
_D_TABLE = [
    [0,1,2,3,4,5,6,7,8,9],
    [1,2,3,4,0,6,7,8,9,5],
    [2,3,4,0,1,7,8,9,5,6],
    [3,4,0,1,2,8,9,5,6,7],
    [4,0,1,2,3,9,5,6,7,8],
    [5,9,8,7,6,0,4,3,2,1],
    [6,5,9,8,7,1,0,4,3,2],
    [7,6,5,9,8,2,1,0,4,3],
    [8,7,6,5,9,3,2,1,0,4],
    [9,8,7,6,5,4,3,2,1,0],
]
_P_TABLE = [
    [0,1,2,3,4,5,6,7,8,9],
    [1,5,7,6,2,8,3,0,9,4],
    [5,8,0,3,7,9,6,1,4,2],
    [8,9,1,6,0,4,3,5,2,7],
    [9,4,5,3,1,2,6,8,7,0],
    [4,2,8,6,5,7,3,9,0,1],
    [2,7,9,3,8,0,6,4,1,5],
    [7,0,4,6,9,1,3,2,5,8],
]


def validate_aadhaar_number(aadhaar_number):
    """
    Validate via Verhoeff checksum.
    Also cleans OCR mis-reads (O→0, I/l→1, S→5, B→8) before checking.
    """
    if aadhaar_number is None:
        return False

    # OCR noise cleanup
    ocr_fix = str.maketrans("OIlSB", "01158")
    cleaned = re.sub(r'[\s\-]', '', str(aadhaar_number).translate(ocr_fix))

    if not cleaned.isdigit():
        return False
    if len(cleaned) != 12:
        return False
    if cleaned[0] in ('0', '1'):   # Aadhaar never starts with 0 or 1
        return False

    c = 0
    for i, digit in enumerate(reversed(cleaned)):
        c = _D_TABLE[c][_P_TABLE[i % 8][int(digit)]]
    return c == 0


def aadhaar_validation_report(aadhaar_data):
    number   = aadhaar_data.get("aadhaar_number")
    is_valid = validate_aadhaar_number(number)
    return {
        "aadhaar_number":  number,
        "checksum_valid":  is_valid,
        "fraud_signal":    not is_valid,
    }


In [23]:
validation_report = aadhaar_validation_report(aadhaar_data)
aadhaar_data["aadhaar_checksum_valid"] = validation_report["checksum_valid"]
print("Validation report:", validation_report)

Validation report: {'aadhaar_number': '7445 3447 0565', 'checksum_valid': True, 'fraud_signal': False}


# **Address Parsing**

In [24]:
def clean_address(raw_address):
    if not raw_address:
        return None
    addr = raw_address.replace('\n', ' ')
    addr = re.sub(r'\s+', ' ', addr)
    # Remove leading 'Address :' label
    addr = re.sub(r'^(addr(ess)?\s*:?\s*)', '', addr, flags=re.IGNORECASE)
    return addr.strip()


def extract_guardian_details(address):
    """
    Matches S/O, D/O, W/O, C/O with and without slash, colon, spaces.
    Also handles OCR splits like 'S / O', 'so:', 'Son of'.
    Guardian name is clipped at common delimiters (comma, digit, keyword).
    """
    if not address:
        return {"guardian_type": None, "guardian_name": None}

    patterns = [
        (r'S[\s/]*O[\s:]+([A-Za-z][A-Za-z .]+)',  'S/O'),
        (r'D[\s/]*O[\s:]+([A-Za-z][A-Za-z .]+)',  'D/O'),
        (r'W[\s/]*O[\s:]+([A-Za-z][A-Za-z .]+)',  'W/O'),
        (r'C[\s/]*O[\s:]+([A-Za-z][A-Za-z .]+)',  'C/O'),
        (r'Son\s+of[\s:]+([A-Za-z][A-Za-z .]+)',  'S/O'),
        (r'Daughter\s+of[\s:]+([A-Za-z][A-Za-z .]+)', 'D/O'),
        (r'Wife\s+of[\s:]+([A-Za-z][A-Za-z .]+)',     'W/O'),
    ]

    for pattern, gtype in patterns:
        m = re.search(pattern, address, re.IGNORECASE)
        if m:
            name = m.group(1).strip()
            # Clip at first comma, digit or short noise word
            name = re.split(r'[,\d]', name)[0].strip()
            # Remove trailing noise words
            noise = r'\b(house|flat|plot|village|vill|po|ps|dist|pin)\b'
            name = re.split(noise, name, flags=re.IGNORECASE)[0].strip()
            return {"guardian_type": gtype, "guardian_name": name}

    return {"guardian_type": None, "guardian_name": None}


def extract_pincode(address):
    if not address:
        return None
    # Look for 6-digit number that doesn't look like a year (not 19xx/20xx)
    for m in re.finditer(r'\b(\d{6})\b', address):
        candidate = m.group(1)
        if not re.match(r'(19|20)\d{4}', candidate):  # not a year
            return candidate
    return None


def extract_city_state(address):
    """
    Best-effort: find lines near the pincode that may be city/state.
    Returns whatever follows the last comma before the pincode.
    """
    if not address:
        return None, None
    # Split on comma and look for segments around pincode
    parts = [p.strip() for p in re.split(r'[,\n]', address)]
    pincode = extract_pincode(address)
    city, state = None, None
    for i, part in enumerate(parts):
        if pincode and pincode in part:
            if i >= 1:
                city  = parts[i-1]
            if i >= 2:
                state = parts[i-2]
            break
    return city, state


def process_address(raw_address):
    cleaned  = clean_address(raw_address)
    guardian = extract_guardian_details(cleaned)
    pincode  = extract_pincode(cleaned)
    city, state = extract_city_state(cleaned)

    return {
        "cleaned_address": cleaned,
        "guardian_type":   guardian["guardian_type"],
        "guardian_name":   guardian["guardian_name"],
        "city":            city,
        "state":           state,
        "pincode":         pincode,
    }


In [25]:
address_info = process_address(aadhaar_data["address"])
print("Address breakdown:", address_info)

aadhaar_data.update(address_info)
del aadhaar_data["address"]

Address breakdown: {'cleaned_address': 'Acaress: SO:Padcep Kumar = ५३५/२. randhawa voad nard no ७ Kharar PO: Kharar DIST: SAS Nagar Mohali). Puniab 140301', 'guardian_type': 'S/O', 'guardian_name': 'Padcep Kumar', 'city': None, 'state': None, 'pincode': '140301'}


In [26]:
print("\n========== FINAL AADHAAR DATA ==========\n")
print(json.dumps(
    {k: v for k, v in aadhaar_data.items() if k not in ('front_text', 'back_text')},
    indent=2, default=str
))


========== FINAL AADHAAR DATA ==========

{
  "name": "Shreshth Garg",
  "dob": "13/07/2005",
  "gender": "Male",
  "aadhaar_number": "7445 3447 0565",
  "photo_present": true,
  "qr_present": {
    "qr_present": true,
    "method": "morphological_corner(TR)",
    "density": 0.352,
    "lap_var": 111.3
  },
  "aadhaar_checksum_valid": true,
  "cleaned_address": "Acaress: SO:Padcep Kumar = \u096b\u0969\u096b/\u0968. randhawa voad nard no \u096d Kharar PO: Kharar DIST: SAS Nagar Mohali). Puniab 140301",
  "guardian_type": "S/O",
  "guardian_name": "Padcep Kumar",
  "city": null,
  "state": null,
  "pincode": "140301"
}
